# Linear Algebra

That linear algebra is fun, is a widely accepted fact. This notebooks will guide you through some of the linear algebra fun you can realize with Heat. 

In [ ]:
from ipyparallel import Client
rc = Client(profile="default")
rc.ids

if len(rc.ids) == 0:
    print("No engines found")
else:
    print(f"{len(rc.ids)} engines found")

4 engines found


In [ ]:
%%px
import heat as ht

%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stderr:3] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


[stderr:2] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


[stderr:1] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


[stderr:0] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/_config.py:108: UserWarning: Heat has CUDA GPU-support (PyTorch version 2.9.1+cu128 and `torch.cuda.is_available() = True`), but CUDA-awareness of MPI could not be detected. This may lead to performance degradation as direct MPI-communication between GPUs is not possible.
  warnings.warn(
/p/software/default/stages/2026/software/zarr/3.1.5-gcccoreflexiblas-14.3.0-3.4.5/lib/python3.13/site-packages/google_crc32c/__init__.py:29: RuntimeWarning: As the c extension couldn't be imported, `google-crc32c` is using a pure python implementation that is significantly slower. If possible, please configure a c build environment and compile the extension
  warnings.warn(_SLOW_CRC32C_WARNING, RuntimeWarning)


Before we dive into it, we specify the device we want to run this tutorial on.
If you have GPUs available, use `device='gpu'`, otherwise, use `device='cpu'`.
Both gives the same results, just at different performance.
For the small examples here, CPUs will do alright, so don't worry if this is all you have right now.

In [ ]:
%%px
ht.use_device('gpu')

## Matrix-Matrix Multiplication

The most basic operation in linear algebra is matrix-matrix multiplication ("matmul"). Doing it by hand for a small matrix is not difficult and in fact not very spectacular. However, in the distributed setting (e.g., on 4 GPUs) even such a simple operation is not trivial any more: just imagine you work together with 3 other people and each of you only knows one fourth of the columns of a matrix $A$ and one fourth of the rows of a matrix $B$. Together, you have to compute the product $AB$ such that in the end each of you only has one fourth of the columns of $AB$...

In [ ]:
%%px
A = ht.random.randn(10000, 10000, split=0)
B = ht.random.randn(A.shape[1], 10000, split=1)
C = ht.matmul(A, B)  # or A @ B
C

Out[1:40]: 
DNDarray([[-1.1320e+02,  3.5233e+01,  1.4170e+02,  ...,  9.5836e+01,  6.4040e+01, -1.2781e+02],
          [-7.6130e+00,  1.1424e+01, -4.8484e+01,  ...,  4.5809e+00,  1.0849e+02,  1.0368e+02],
          [-6.5867e+00, -1.3176e+02,  1.4061e+02,  ..., -8.8871e+01,  1.5092e+01,  1.5137e+02],
          ...,
          [-6.2212e+00,  1.4790e+01, -3.5197e+01,  ..., -5.8485e-02, -1.6385e+02, -4.3784e+01],
          [-8.3889e+01,  1.8936e+02, -4.4213e+01,  ...,  1.6859e+01, -5.9910e+01, -2.1940e+01],
          [-5.3905e+01, -9.5783e+01, -1.7910e+02,  ..., -7.8864e+01,  2.3121e+01,  1.2461e+02]], dtype=ht.float32, device=gpu:0, split=0)

## QR Decomposition and Triangular Solve

Given a matrix $A$, its QR decomposition is given by $A=QR$ where $Q$ is an orthogonal matrix (i.e. its columns are pairwise orthonormal) and $R$ is an upper triangular matrix. 

Further information: [QR on Wikipedia](https://en.wikipedia.org/wiki/QR_decomposition)

In [ ]:
%%px
A = ht.random.randn(100000, 1000, split=0)
Q,R = ht.linalg.qr(A)
print(f'{Q.shape=} {Q.split=} {Q.device=} | {R.shape=} {R.split=} {R.device=}')

[stdout:1] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


[stdout:2] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


[stdout:3] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


[stdout:0] Q.shape=(100000, 1000) Q.split=0 Q.device=device(gpu:0) | R.shape=(1000, 1000) R.split=None R.device=device(gpu:0)


In [ ]:
%%px
print(Q.T @ Q)
print(f'Largest off-diagonal value: {ht.linalg.norm(Q.T @ Q - ht.eye(Q.shape[1]), ord=ht.inf).item():.2e}')

[stdout:2] 
Largest off-diagonal value: 3.48e-06


[stdout:0] 
Largest off-diagonal value: 3.48e-06


[stdout:3] 
Largest off-diagonal value: 3.48e-06


[stdout:1] DNDarray([[ 1.0000e+00, -4.4238e-09,  1.8917e-09,  ..., -6.4028e-10, -3.2596e-09, -1.8626e-09],
          [-4.4238e-09,  1.0000e+00, -2.3283e-10,  ..., -3.7253e-09,  6.5775e-09, -4.8894e-09],
          [ 1.8917e-09, -2.3283e-10,  1.0000e+00,  ...,  1.2806e-09, -2.1537e-09,  4.6566e-10],
          ...,
          [ 2.0955e-09, -2.5611e-09,  8.1491e-10,  ...,  1.0000e+00,  4.9768e-09,  5.8208e-11],
          [ 9.3132e-10,  3.8417e-09,  1.5716e-09,  ...,  4.9768e-09,  1.0000e+00,  1.8044e-09],
          [-9.3132e-10, -3.7253e-09,  2.3283e-09,  ...,  5.8208e-11,  1.8044e-09,  1.0000e+00]], dtype=ht.float32, device=gpu:0, split=1)
Largest off-diagonal value: 3.48e-06


In [ ]:
%%px
print(R)

[stdout:2] 


[stdout:0] 


[stdout:3] 


[stdout:1] DNDarray([[ 3.1581e+02,  1.9340e+00, -4.7852e-02,  ..., -4.0572e-01, -1.0129e+00, -3.9183e-01],
          [ 0.0000e+00,  3.1659e+02,  1.5766e+00,  ...,  1.5641e+00, -2.0920e-01,  1.1468e+00],
          [ 0.0000e+00,  0.0000e+00, -3.1584e+02,  ..., -1.3541e+00,  1.6570e+00,  1.4314e-01],
          ...,
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  3.1512e+02,  1.7609e-01, -1.7567e-01],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00, -3.1603e+02,  7.0108e-02],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,  0.0000e+00, -3.1455e+02]], dtype=ht.float32, device=gpu:0, split=None)


In [ ]:
%%px
norm_A = ht.linalg.norm(A)
print(f'Reconstruction error: {ht.linalg.norm(Q@R - A)/norm_A).item()}, Machine precision: {ht.types.finfo(A.dtype).eps}')

[stdout:3] Reconstruction error: 2.6260e-07, Machine precision: 1.1920928955078125e-07


[stdout:2] Reconstruction error: 2.6260e-07, Machine precision: 1.1920928955078125e-07


[stdout:0] Reconstruction error: 2.6260e-07, Machine precision: 1.1920928955078125e-07


[stdout:1] Reconstruction error: 2.6260e-07, Machine precision: 1.1920928955078125e-07


We can see that the reconstruction error is actually very close to machine precision in Heat's default datatype.

With a little bit of linear algebra fun, you find out that a linear least squares problem of type $\min \lVert Ax - b \rVert_2$ boils down to computing the QR decomposition $A=QR$ and then solving for $Rx = Q^T b$. (Of course, we need to assume that if $A \in \mathbb{R}^{m \times n}$ that $m \geq n$ and $R$ is invertible...)

In [ ]:
%%px
b = ht.random.randn(100000,1,split=None)
Qtb = Q.T @ b
x = ht.linalg.solve_triangular(R,Qtb.resplit_(0)) 
print(f'Residual at x: {float(ht.norm(A @ x - b)/norm_A):.2e}')
print(f'Gradient of squared problem at x: {float(ht.norm(A.T @ A @ x - A.T @ b)/norm_A):.2e}')

[stdout:2] Residual at x: 3.14e-02
Gradient of squared problem at x: 5.89e-07


[stdout:1] Residual at x: 3.14e-02
Gradient of squared problem at x: 5.89e-07


[stdout:0] Residual at x: 3.14e-02
Gradient of squared problem at x: 5.89e-07


[stdout:3] Residual at x: 3.14e-02
Gradient of squared problem at x: 5.89e-07


If you want to solve a LASSO-regularized version of this linear regression problem, try out `heat.regression.Lasso`!

## Singular Value Decomposition

Let $X \in \mathbb{R}^{m \times n}$ be a matrix, e.g., given by a data set consisting of $m$ data points $\in \mathbb{R}^n$ stacked together. The so-called **singular value decomposition (SVD)** of $X$ is given by 

$$
	X = U \Sigma V^T
$$

where $U \in \mathbb{R}^{m \times r_X}$ and $V \in \mathbb{R}^{n \times r_X}$ have orthonormal columns, $\Sigma = \text{diag}(\sigma_1,...,\sigma_{r_X}) \in \mathbb{R}^{r_X \times r_X}$ is a diagonal matrix containing the so-called singular values $\sigma_1 \geq \sigma_2 \geq ... \geq \sigma_{r_X} > 0$, and $r_X \leq \min(m,n)$ denotes the rank of $X$ (i.e. the dimension of the subspace of $\mathbb{R}^m$ spanned by the columns of $X$). Since $\Sigma = U^T X V$ is diagonal, one can imagine this decomposition as finding orthogonal coordinate transformations under which $X$ looks "linear". Further information: [SVD on Wikipedia](https://en.wikipedia.org/wiki/Singular_value_decomposition)

Computing the **full** SVD in a distributed environment can be quite **expensive**; nevertheless, we have implemented it: 

In [ ]:
%%px
A = ht.random.rand(2000,2000, split=1)
U, S, VT = ht.linalg.svd(A)
print(U, S, VT)

[stderr:1] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


[stderr:2] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


[stderr:0] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


[stderr:3] /p/project1/ccstma/baumann7/heat/doc/source/tutorials/notebooks/.venv/lib/python3.13/site-packages/heat/core/linalg/svd.py:184: UserWarning: You are performing the full SVD of a distributed matrix that is neither of tall-skinny nor short-fat shape. 
 This operation may be costly in terms of memory and compute time.
  warn(


%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stdout:0]   


[stdout:2]   


[stdout:3]   


[stdout:1] DNDarray([[-2.1918e-02,  6.8974e-03, -1.7709e-02,  ...,  7.9259e-03, -2.5663e-02, -4.0795e-03],
          [-2.2347e-02,  1.4844e-03, -1.1549e-02,  ...,  4.4652e-02, -1.4302e-02,  4.3925e-03],
          [-2.2653e-02,  7.2853e-04, -2.3686e-02,  ..., -3.1769e-02,  5.2439e-03,  1.3047e-02],
          ...,
          [-2.3093e-02,  1.0162e-02,  4.5365e-02,  ...,  3.7869e-03, -3.7170e-02,  9.9250e-03],
          [-2.2156e-02, -4.2652e-03, -5.7889e-02,  ...,  1.8692e-02,  1.1928e-02, -1.6929e-02],
          [-2.2554e-02, -5.1398e-02,  2.4149e-02,  ...,  1.5852e-02,  2.3505e-02,  1.3751e-02]], dtype=ht.float32, device=gpu:0, split=1) DNDarray([9.9990e+02, 2.5770e+01, 2.5580e+01,  ..., 9.8730e-03, 4.8357e-03, 3.0352e-03], dtype=ht.float32, device=gpu:0, split=0) DNDarray([[-2.2477e-02, -2.2122e-02, -2.2107e-02,  ..., -2.2150e-02, -2.2842e-02, -2.1813e-02],
          [ 1.6411e-02,  2.7478e-02,  3.7353e-03,  ...,  5.9210e-03,  8.5628e-03, -4.5939e-02],
          [-4.0299e-03, -8.2720e-0

In [ ]:
%%px
norm_A = ht.linalg.norm(A, ord=ht.inf)
print(ht.linalg.norm(U.T @ U - ht.eye(U.shape[1]), ord=ht.inf) / norm_A)

[stdout:2] 


[stdout:1] DNDarray([2.7522e-06], dtype=ht.float32, device=gpu:0, split=None)


[stdout:3] 


[stdout:0] 


In [ ]:
%%px
print(ht.linalg.norm(VT.T @ VT - ht.eye(VT.shape[1]), ord=ht.inf) / norm_A)

[stdout:0] 


[stdout:1] DNDarray([3.2494e-06], dtype=ht.float32, device=gpu:0, split=None)


[stdout:2] 


[stdout:3] 


In [ ]:
%%px
print(ht.norm(A - U @ ht.diag(S) @  VT, ord=ht.inf)/norm_A)

[stdout:2] 


[stdout:3] 


[stdout:0] 


[stdout:1] DNDarray([0.0005], dtype=ht.float32, device=gpu:0, split=None)


If the number of rows is much higher than the number of columns (we call such matrices "tall-skinny"), a more efficient implementation of SVD is available than in the general case:

In [ ]:
%%px 
X = ht.random.rand(100000,2000, split=0)
U, S, VT = ht.linalg.svd(X)
norm_X = ht.linalg.norm(X, ord=ht.inf)
print(ht.linalg.norm(U.T @ U - ht.eye(U.shape[1]), ord=ht.inf) / norm_X)
print(ht.linalg.norm(VT @ VT.T - ht.eye(VT.shape[0]), ord=ht.inf) / norm_X)
print(ht.norm(X - U @ ht.diag(S) @  VT, ord=ht.inf) / norm_X)

[stdout:3] 




[stdout:0] 




[stdout:1] DNDarray([4.3423e-06], dtype=ht.float32, device=gpu:0, split=None)
DNDarray([1.7939e-06], dtype=ht.float32, device=gpu:0, split=None)
DNDarray([0.0003], dtype=ht.float32, device=gpu:0, split=None)


[stdout:2] 




## Truncated SVD 

So, what to do if your matrix is very large but not tall-skinny? 

Let us start with a brief background on how SVD is actually used in applications... In data science, SVD is more often known as **principle component analysis (PCA)**, the columns of $U$ being called the principle components of $X$. In fact, in many applications **truncated SVD/PCA** suffices: to reduce $X$ to the "essential" information, one chooses a truncation rank $0 < r \leq r_X$ and considers the truncated SVD/PCA given by 

$$
X \approx X_r := U_{[:,:r]} \Sigma_{[:r,:r]} V_{[:,:r]}^T
$$

where we have used `numpy`-like notation for selecting only the first $r$ columns of $U$ and $V$, respectively. The rationale behind this is that if the first $r$ singular values of $X$ are much larger than the remaining ones, $X_r$ will still contain all "essential" information contained in $X$; in mathematical terms: 

$$
\lVert X_r - X \rVert_{F}^2 = \sum_{i=r+1}^{r_X} \sigma_i^2, 
$$

where $\lVert \cdot \rVert_F$ denotes the Frobenius norm. Thus, truncated SVD/PCA may be used for, e.g.,  
* filtering away non-essential information in order to get a "feeling" for the main characteristics of your data set, 
* to detect linear (or "almost" linear) dependencies in your data, 
* to generate features for further processing of your data. 

In the following we present two options available in Heat to compute an **approximate truncated SVD for a large matrix**. 

### Randomized SVD 

The most easy option is so-called randomized SVD which allows you to specify the rank $r$ you are targeting as second positional argument. Here, we use randomized SVD with a certain number of oversamples and one power iteration in order to compute an approximation to the leading 10 singular values and vectors of the same matrix $A$ as above:   

In [ ]:
%%px 
Ur, Sr, Vr = ht.linalg.rsvd(X, 10, n_oversamples=10, power_iter=1)
print(Ur)
print(Sr)
print(Vr)

%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stdout:3] 




[stdout:2] 




[stdout:0] 




[stdout:1] DNDarray([[-3.1279e-03,  1.1720e-03,  5.4030e-03,  ..., -2.4220e-03, -4.0192e-03,  5.5848e-04],
          [-3.1227e-03, -8.1526e-04, -1.1014e-03,  ..., -3.2516e-03, -1.5570e-03, -1.4586e-03],
          [-3.1452e-03,  1.3700e-03,  3.3549e-03,  ..., -4.1605e-03, -1.8204e-03, -8.9519e-04],
          ...,
          [-3.1585e-03, -2.7211e-03,  1.4667e-03,  ...,  8.5815e-04,  2.3877e-03,  2.7780e-03],
          [-3.0894e-03, -2.0493e-03,  5.1568e-05,  ..., -3.2446e-03, -5.1416e-03,  6.3745e-03],
          [-3.1838e-03, -3.7771e-03, -6.5672e-03,  ...,  1.0414e-03, -1.2793e-03, -4.5323e-04]], dtype=ht.float32, device=gpu:0, split=0)
DNDarray([7072.1885,   95.0284,   94.7281,   94.6208,   94.5116,   94.4345,   94.2615,   94.1368,   94.1010,   93.8940], dtype=ht.float32, device=gpu:0, split=None)
DNDarray([[-0.0224, -0.0224, -0.0224,  ..., -0.0224, -0.0224, -0.0223],
          [-0.0005,  0.0064,  0.0440,  ..., -0.0279, -0.0416, -0.0015],
          [-0.0343,  0.0159, -0.0337,  ..., -0.

### Hierarchical SVD 

Hierarchical SVD allows a bit more of freedom. Truncation takes place either w.r.t. a fixed truncation-rank (`heat.linalg.hsvd_rank`) or w.r.t. a desired accuracy (`heat.linalg.hsvd_rtol`). In the latter case it can be ensured that it holds for the "reconstruction error": 

$$
\frac{\lVert X - U U^T X \rVert_F}{\lVert X \rVert_F} \overset{!}{\leq} \text{rtol},
$$

where $U$ denotes the approximate left-singular vectors of $X$ computed by `heat.linalg.hsvd_rtol`. 

Let's first compute the truncated SVD by setting the relative tolerance: 

In [ ]:
%%px
# compute truncated SVD w.r.t. relative tolerance 
svd_with_reltol = ht.linalg.hsvd_rtol(X,rtol=1.0e-2,compute_sv=True,silent=False)
U = svd_with_reltol[0]
reconstruction_error = (ht.linalg.norm(X - U@U.T@X, ord='fro') / ht.linalg.norm(X, ord='fro')).item()
print(f"{reconstruction_error=} relative residual: {svd_with_reltol[3].item()} rank: {U.shape[1]}")

[stdout:1] hSVD level 0...	 processes  0		1		2		3
              current ranks: 2000		2000		2000		2000
hSVD level 1...	 processes  0		2
              current ranks: 2000		2000
hSVD level 2...	 processes  0
reconstruction_error=0.00084589485777542 relative residual: 0.026991277933120728 rank: 2000


%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

[stdout:0] reconstruction_error=0.00084589485777542 relative residual: 0.026991277933120728 rank: 2000


[stdout:2] reconstruction_error=0.00084589485777542 relative residual: 0.026991277933120728 rank: 2000


[stdout:3] reconstruction_error=0.00084589485777542 relative residual: 0.026991277933120728 rank: 2000


Alternatively, you can compute a truncated SVD with a fixed truncation rank:

In [ ]:
%%px
# compute truncated SVD w.r.t. a fixed truncation rank 
svd_with_rank = ht.linalg.hsvd_rank(X, maxrank=3,compute_sv=True,silent=False)
U = svd_with_rank[0]
reconstruction_error = (ht.linalg.norm(X - U@U.T@X, ord='fro') / ht.linalg.norm(X, ord='fro')).item()
print(f"{reconstruction_error=} relative residual: {svd_with_rank[3].item()} rank: {U.shape[1]}")

[stdout:1] hSVD level 0...	 processes  0		1		2		3
              current ranks: 8		8		8		8
hSVD level 1...	 processes  0
reconstruction_error=0.49953070282936096 relative residual: 0.5008435845375061 rank: 3


[stdout:0] reconstruction_error=0.49953070282936096 relative residual: 0.5008435845375061 rank: 3


[stdout:3] reconstruction_error=0.49953070282936096 relative residual: 0.5008435845375061 rank: 3


[stdout:2] reconstruction_error=0.49953070282936096 relative residual: 0.5008435845375061 rank: 3


Once we have computed the truncated SVD, we can use it to approximate the original data matrix `X` by the truncated matrix `X_r`. 

Check out the plot below to see how Heat's truncated SVD algorithm scales with the number of MPI processes and size of the dataset.

<div>
  <img src=https://github.com/helmholtz-analytics/heat/blob/main/doc/source/_static/images/hSVD_bench_rank5.png?raw=true title="hSVD rank 5" width="30%" style="float:left"/>
  <img src=https://github.com/helmholtz-analytics/heat/blob/main/doc/source/_static/images/hSVD_bench_rank50.png?raw=true title="hSVD rank 50" width="30%" style="float:center "/>
  <img src=https://github.com/helmholtz-analytics/heat/blob/main/doc/source/_static/images/hSVD_bench_rank500.png?raw=true title="HSVD rank 500" width="30%" style="float:center"/>
</div>

## Exercises 

1. Try out different split combinations `split_A=0,1,None`, `split_B=0,1,None` for matrix-matrix multiplication. What do you observe for the split of the outcome? Does every combination take the same computing time? (Whats the likely cause for this?)

2. Compare the approximate singular values computed by randomized SVD / hierarchical SVD with the reference ones computed by full SVD. What do you observe for different choices of the parameters (number of oversamples, power iterations, truncation rank, truncation tolerance)? 